In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Backtesting with derived portfolios

Shows how to use a derived portfolio to test different trading strategies.

Attributes
----------
cocoon
instruments
holdings
quotes
aggregation
derived portfolios
"""

toggle_code("Toggle Docstring")

# Backtesting strategy selection on PnL 

The aim of this notebook is to show strategy backtesting using LUSID's derived portfolio functionality. In this notebook we will compare the performance of a US technology sector equity portfolio under two seperate scenarios. In the first scenario, the portfolio will be weighted towards "Google". In the second scenario, the portfolio will be weighted towards "Netflix". We will then compare PnL results. 

From a LUSID perspective, we will complete the following:

(1) Setup the LUSID environment. <br>

(2) Create our first portfolio. <br>

(3) Load our FANG (Facebook, Apple, Netflix, Google) instrument master. <br>

(4) Upsert initial holdings - the portfolio will have a 55% of NAV weighting to "Google". <br>

(5) Upsert quotes to cover a 9 month period (1 Jan 2019 to 30 Sep 2019). <br>

(6) Run aggregration on Google heavy portfolio. <br>

(7) Plot the results of this portfolio on a graph. <br>

(8) Did we make the correct allocation decision? By using LUSID's derived portfolios, we can wind back time and check performance of the portfolio if we allocated 55% to Netflix instead of Google. <br>

(9) Plot and compare time series. <br>


# Step 1: Import packages and initialize LUSID environment

In [ ]:
# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

# import lusid_sample_data as import_data
from finbourne_sdk_utils.cocoon import cocoon
from finbourne_sdk_utils.cocoon import instruments as cocoon_instruments

# Import Libraries
import pprint
from datetime import datetime, timedelta, time, date
import pytz
import uuid
import os
import printer as prettyprint
from datetime import datetime
import pandas as pd
import numpy as np
import lusid_sample_data as import_data
import json

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

print("LUSID Environment Initialised")
print(
    "LUSID SDK Version: ",
    api_factory.build(lu.ApplicationMetadataApi)
    .get_lusid_versions()
    .build_version,
)

In [ ]:
# Import ploting libraries and configuration
import matplotlib.pyplot as plt

# Use line magic function to enable matplotlib to work interactively with iPython

%matplotlib inline

# Set style to fivethirtyeight to create clean and clear looking graphs

plt.style.use("fivethirtyeight")

# Define a dictionary containing default plotting configurations

params = {
    "legend.fontsize": "small",
    "figure.figsize": (12, 4.5),
    "axes.labelsize": "small",
    "axes.titlesize": "medium",
    "xtick.labelsize": "small",
    "ytick.labelsize": "small",
}

plt.rcParams.update(params)

Define two portfolios:<br>

(1) portfolioA will be used to test performance when 55% of the portfolio is weighted to Google. <br>

(2) portfolioB will be used to test performance when 55% of the portfolio is weighted to Netflix. <br>

In [ ]:
portfolioA = "US_TECH1"
portfolioB = "US_TECH2"

Define a scope.

A scope in LUSID is a partioning of data. See our support page for [Scopes](https://support.lusid.com/what-is-a-scope-in-lusid-and-how-is-it-used) for more details:



In this notebook, we will use the scope to partition portfolios, instruments, quotes and holdings data.



In [ ]:
scope = "techStrategy"

# Step 2: Create the first portfolio

To get started, we first need to load a portfolio. The portfolio will be used to store holdings.

In [ ]:
# Load a CSV file of portfolio data into a pandas DataFrame

portfolio_csv = r"data/backtesting/backtesting_portfolios.csv"
df_portfolios_csv = pd.read_csv(portfolio_csv)
df_portfolios_csv["portfolio_ticker"] = portfolioA
df_portfolios_csv

In [ ]:
# Create a dictionary of mappings
# The dictionary keys are the names for values in LUSID
# The dictionary values are the names in our CSV column headers

mapping_required = {
    "display_name": "portfolio_name",
    "code": "portfolio_ticker",
    "base_currency": "base_currency",
}


mapping_optional = {"description": "portfolio_name", "created": "created"}

In [ ]:
# Use the load_from_data_frame method from LUSID's Python cocoon package to upload the portfolio

response = cocoon.load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df_portfolios_csv,
    property_columns=["location"],
    mapping_required=mapping_required,
    mapping_optional=mapping_optional,
    file_type="portfolios",
)

# Step 3: Upsert our instrument master

Next we upsert some instruments into LUSID. Simliar to the above, the instruments will be created from a CSV file.

In [ ]:
# Load an instruments file from CSV
# The details will be stored in a Pandas DataFrame

instruments_csv = r"data/backtesting/backtesting_instruments.csv"
df_instruments = pd.read_csv(instruments_csv)
df_instruments

In [ ]:
# Create dictionaries of mappings

mapping_required = {
    "name": "name",
}

# This time, we also need to tell LUSID about our unique identifiers
# All instruments in LUSID need a unique identifier

identifiers = {"ClientInternal": "ClientInternal", "Ticker": "Ticker"}

In [ ]:
# Call cocoon again
# But this time we use a file_type of "instruments" to call the instruments endpoint

response = cocoon.load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df_instruments,
    mapping_required=mapping_required,
    mapping_optional={},
    file_type="instruments",
    identifier_mapping=identifiers,
)

# Step 4: Create initial holdings for 1 Jan 2019

Now that we have a portfolio and some instruments, we want to create holdings in that portfolio using those instruments.

In [ ]:
# Load holdings into DataFrame

holdings_csv = r"data/backtesting/backtesting_holdings.csv"
df_holdings = pd.read_csv(holdings_csv)
df_holdings["effective_at"] = "2019-01-01T00:00:00Z"
df_holdings["portfolio_code"] = portfolioA
df_holdings["market_value"] = df_holdings["quantity"] * df_holdings["price"]
df_holdings

In [ ]:
def pie_plot():
    plt.style.use("fivethirtyeight")
    pie_plot = df_holdings.plot.pie(
        y="market_value",
        labels=df_holdings["instrument_name"].values,
        figsize=(11, 11),
        title=f"Allocation of portfolio {portfolioA} to FANG stocks by % of NAV",
        explode=(0.025, 0, 0, 0),
        startangle=140,
        autopct="%1.1f%%",
        textprops={"fontsize": 16},
    )

    pie_plot.set_ylabel("")
    pie_plot.legend(loc=4, prop={"size": 14})
    pie_plot.set_title(
        f"Allocation of portfolio {portfolioA} to FANG stocks by % of NAV", fontsize=20
    )

In [ ]:
pie_plot()

In [ ]:
# Provide mappings (as above)
# Adding a $ prefix to the dict value will hardcode that value into the LUSID call
# In the example below, we only want to set holdings, we are not concerned about the cash impact.
# Therefore we set the portfolio_cost and cost.amount to 0

mapping_required = {
    "name": "instrument_name",
    "effective_at": "effective_at",
    "code": "portfolio_code",
    "tax_lots.units": "quantity",
    "tax_lots.price": "price",
    "tax_lots.portfolio_cost": "market_value",
    "tax_lots.cost.currency": "currency",
    "tax_lots.cost.amount": "market_value",
}

identifiers = {
    "ClientInternal": "ClientInternal",
}

In [ ]:
# Run the load_from_data_frame cocoon method again
# This time we upload holdings

response = cocoon.load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df_holdings,
    mapping_required=mapping_required,
    mapping_optional={},
    file_type="holdings",
    identifier_mapping=identifiers,
)

# Step 5: Upload quotes

Next we need some quotes, to produce our valuations. In this example, the quotes will be loaded from an external CSV file.

In [ ]:
quotes_csv = r"data/backtesting/backtesting_quotes.csv"
df_quotes = pd.read_csv(quotes_csv)
df_quotes.head(2)

In [ ]:
df_quotes.loc[df_quotes["quote_date"] == "2019-01-02T00:00:00Z"]

We now have the quotes in a DataFrame, which is great. However we also want to provide the LUID. To resolve this problem, we take our client_internal ID and search the APIs for a LUID which is appended to each row in the DataFrame. <br>

In [ ]:
def add_luid_id(data_frame):

    client_ids = pd.DataFrame(
        list(df_holdings["ClientInternal"].unique()), columns=["client_internal"]
    )
    client_ids["LUID"] = client_ids["client_internal"].apply(
        lambda x: api_factory.build(lu.InstrumentsApi)
        .get_instrument(identifier_type="ClientInternal", identifier=x)
        .lusid_instrument_id
    )

    client_ids = client_ids.set_index("client_internal")

    data_frame["LUID"] = data_frame["client_internal"].apply(
        lambda x: client_ids.loc[x]["LUID"]
    )

    return data_frame

In [ ]:
df_quotes = add_luid_id(df_quotes)

In [ ]:
# Define our mapping dictionaries

mapping_required = {
    "quote_id.quote_series_id.instrument_id_type": "$LusidInstrumentId",
    "quote_id.effective_at": "quote_date",
    "quote_id.quote_series_id.provider": "$DataScope",
    "quote_id.quote_series_id.var_field": "$mid",
    "quote_id.quote_series_id.quote_type": "$Price",
    "quote_id.quote_series_id.instrument_id": "LUID",
    "metric_value.unit": "$USD",
    "metric_value.value": "price",
}

In [ ]:
# Run cocoon again, this time for quotes

response = cocoon.load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df_quotes,
    mapping_required=mapping_required,
    mapping_optional={},
    file_type="quotes",
)

prettyprint.upsert_quotes_response(response["quotes"]["success"][0]).head(10)

# Step 6: Run aggregration on portfolio

Now we have all our data in LUSID to produce a valuation. Let's see how well our portfolio performed between 1 Jan and 30 Sep. Was it a good idea to allocate 55% to Google?

In [ ]:
recipe_code = "market_value"
recipe_scope = "backtesting"

# Create a recipe to perform a valuation
configuration_recipe_api = api_factory.build(lu.ConfigurationRecipeApi)

configuration_recipe = models.ConfigurationRecipe(
    scope=recipe_scope,
    code=recipe_code,
    market=models.MarketContext(
        market_rules=[
            models.MarketDataKeyRule(
                key="Quote.LusidInstrumentId.*",
                supplier="DataScope",
                data_scope=scope,
                quote_type="Price",
                field="mid",
                # quote_interval=date.strftime("%Y-%m-%d") + ".0D"
                quote_interval="1W",
            )
        ],
        suppliers=models.MarketContextSuppliers(
            commodity="DataScope",
            credit="DataScope",
            equity="DataScope",
            fx="DataScope",
            rates="DataScope",
        ),
        options=models.MarketOptions(
            default_supplier="DataScope",
            default_instrument_code_type="LusidInstrumentId",
            default_scope=scope,
        ),
    ),
)

upsert_configuration_recipe_response = (
    configuration_recipe_api.upsert_configuration_recipe(
        upsert_recipe_request=models.UpsertRecipeRequest(
            configuration_recipe=configuration_recipe
        )
    )
)

In [ ]:
# Create an aggregration function
# This function takes two parameters of data and portfolio_code
# It returns a valuation of the portfolio for the date
# The return is a list of format [portfolio, date, valuation]
# Example [TECH_PORTFOLIO, 2019-10-10, 10000000]


def run_agg(start_date, end_date, portfolio_code):

    # Create the valuation request
    valuation_request = models.ValuationRequest(
        recipe_id=models.ResourceId(scope=recipe_scope, code=recipe_code),
        metrics=[
            models.AggregateSpec(key="Analytic/default/ValuationDate", op="Value"),
            models.AggregateSpec(key="Valuation/PvInReportCcy", op="Sum"),
        ],
        group_by=[
            "Analytic/default/ValuationDate",
        ],
        portfolio_entity_ids=[
            models.PortfolioEntityId(scope=scope, code=portfolio_code)
        ],
        valuation_schedule=models.ValuationSchedule(
            effective_from=start_date.isoformat(), effective_at=end_date.isoformat()
        ),
    )

    # Perform a valuation
    valuation = api_factory.build(lu.AggregationApi).get_valuation(
        valuation_request=valuation_request
    )

    return valuation

In [ ]:
def get_agg_df(agg_data):

    agg_df = (
        pd.DataFrame(agg_data)
        .rename(
            columns={
                "Sum(Valuation/PvInReportCcy)": "MarketValue",
                "Analytic/default/ValuationDate": "Date",
            }
        )
        .sort_values("Date", axis=0)
    )
    agg_df["MarketValue"] = agg_df["MarketValue"] / 1000000
    agg_df["Portfolio"] = portfolioA
    agg_df["Date"] = pd.to_datetime(agg_df["Date"])
    agg_df.set_index("Date", inplace=True)

    return agg_df

In [ ]:
start_date = datetime(year=2019, month=1, day=2, tzinfo=pytz.UTC)
end_date = datetime(year=2019, month=9, day=30, tzinfo=pytz.UTC)

agg = run_agg(start_date, end_date, portfolioA)

In [ ]:
portfolioA_df = get_agg_df(agg.data)
portfolioA_df.head(5)

# Step 7: Plot results of portfolio 

Plot the results of the DataFrame in a nice time-series. We can see our performance was good. The NAV of the portfolio when from 100 million USD on 1 Jan to just under 120 million USD on 30 Sep.

In [ ]:
def time_series_performance():
    ts_performance = portfolioA_df.plot(y=["MarketValue"], figsize=(12, 9))
    ts_performance.set_title(
        f"NAV of portfolio {portfolioA} from 01-01-2019 to 30-09-2019", fontsize="large"
    )
    ts_performance.set_ylabel("NAV in millions of $", fontsize="large")
    ts_performance.set_xlabel("Time: Jan to Sep 2019", fontsize="large")
    ts_performance.legend(prop={"size": 12})

In [ ]:
time_series_performance()

# Step 8: Did we make the right allocation choice?

What would have happened is we allocated 55% of portfolio NAV to Netflix instead of Google. How would that have impacted our portfolio's performance? Fortunately, we can check that easily using LUSID's derived portfolios functionality.<br>

First lets create a derived portfolio from the parent portfolio which is portfolioA above. The derived portfolio will inherit all the holdings of the parent portfolio. See our page on [derived portfolios](https://support.lusid.com/what-is-a-derived-portfolio) to learn more about this portfolio type.

In [ ]:
def create_derived_portfolio(new_port, parent_port):

    derived_api = api_factory.build(lu.DerivedTransactionPortfoliosApi)

    derived_request = models.CreateDerivedTransactionPortfolioRequest(
        display_name=f"Derived Portfolio of {parent_port}",
        code=new_port,
        parent_portfolio_id=models.ResourceId(scope=scope, code=parent_port),
        description="US Tech Strategy",
        created="2018-06-01T00:00:00Z",
    )

    try:

        response = derived_api.create_derived_portfolio(
            scope=scope, create_derived_transaction_portfolio_request=derived_request
        )

        print(response)

    except ApiException as e:
        if e.status == 400 and json.loads(e.body)["code"] == 112:
            print(json.loads(e.body)["title"])

In [ ]:
create_derived_portfolio(portfolioB, portfolioA)

Next let's calculate the holding adjustments required to rebalance 55% of NAV to Netflix.

In [ ]:
def rebalance_port(scope, portfolioA, effective_at, adjustment):

    holdings_response = (
        api_factory.build(lu.TransactionPortfoliosApi)
        .get_holdings(scope, portfolioA, effective_at=effective_at)
        .values
    )

    rebal_holdings = pd.DataFrame(
        [
            [holding.cost_portfolio_ccy.amount, holding.instrument_uid, holding.units]
            for holding in holdings_response
        ],
        columns=["market_value", "LUID", "units"],
    )

    rebal_holdings["client_internal"] = rebal_holdings["LUID"].apply(
        lambda x: api_factory.build(lu.InstrumentsApi)
        .get_instrument(identifier_type="LusidInstrumentId", identifier=x)
        .identifiers["ClientInternal"]
    )

    rebal_holdings["instrument_name"] = rebal_holdings["LUID"].apply(
        lambda x: api_factory.build(lu.InstrumentsApi)
        .get_instrument(identifier_type="LusidInstrumentId", identifier=x)
        .name
    )

    rebal_holdings["Ticker"] = rebal_holdings["LUID"].apply(
        lambda x: api_factory.build(lu.InstrumentsApi)
        .get_instrument(identifier_type="LusidInstrumentId", identifier=x)
        .identifiers["Ticker"]
    )

    rebal_holdings["pct_mv"] = rebal_holdings["market_value"] / sum(
        rebal_holdings["market_value"], 0
    )

    rebal_holdings["target_pct_mv"] = rebal_holdings["pct_mv"]
    for client_internal, value in adjustment:
        rebal_holdings.loc[
            rebal_holdings["client_internal"] == client_internal, "target_pct_mv"
        ] = value

    rebal_holdings["target_units"] = (
        rebal_holdings["target_pct_mv"]
        / rebal_holdings["pct_mv"]
        * rebal_holdings["units"]
    )

    return rebal_holdings

Running the rebalance function below, we can see the new target units for GOOG and NFLX:

In [ ]:
rebalanced_portfolio = rebalance_port(
    scope,
    portfolioA,
    "2019-01-01T00:00:02Z",
    [("EQ38475943", 0.15), ("EQ36852475", 0.55)],
)
rebalanced_portfolio[["Ticker", "target_pct_mv", "target_units"]].set_index("Ticker")

The new % NAVs can be plotted on a pie-chart so we can visualise the results:

In [ ]:
def rebalanced_portfolio_pie():
    plt.style.use("fivethirtyeight")
    rebalanced_portfolio_pie = rebalanced_portfolio.plot.pie(
        y="target_pct_mv",
        labels=rebalanced_portfolio["instrument_name"].values,
        figsize=(11, 11),
        title=f"Allocation of portfolio {portfolioA} to FANG stocks by % of NAV",
        explode=(0.025, 0, 0, 0),
        startangle=140,
        autopct="%1.1f%%",
        textprops={"fontsize": 16},
    )

    rebalanced_portfolio_pie.set_ylabel("")
    rebalanced_portfolio_pie.legend(loc=4, prop={"size": 14})
    rebalanced_portfolio_pie.set_title(
        f"Rebalanced allocation of portfolio {portfolioA} to FANG stocks by % of NAV",
        fontsize=20,
    )

In [ ]:
rebalanced_portfolio_pie()

Once we're happy with the rebalance quantities, these adjustments can be processed in LUSID using the Adjust Holdings APIs:

In [ ]:
def holding_adj_request(holdings):

    holding_adj_request = []

    for instrument, units in holdings:

        holding_adj_request.append(
            models.AdjustHoldingRequest(
                instrument_identifiers={
                    "instrument/default/ClientInternal": f"{instrument}"
                },
                tax_lots=[
                    models.TargetTaxLotRequest(
                        units=units,
                        portfolio_cost=None,
                        cost=models.CurrencyAndAmount(amount=0, currency="USD"),
                        price=None,
                        purchase_date="2019-01-01T00:00:00Z",
                        settlement_date="2019-01-01T00:00:00Z",
                    )
                ],
            )
        )

    return holding_adj_request

In [ ]:
holdings = [("EQ38475943", 14390), ("EQ36852475", 206009)]

holding_adjustment = holding_adj_request(holdings)

In [ ]:
response = api_factory.build(lu.TransactionPortfoliosApi).adjust_holdings(
    scope,
    portfolioB,
    effective_at="2019-01-01T10:00:00Z",
    adjust_holding_request=holding_adjustment,
)

Then rerun our aggregration functions and plot results against each other:

In [ ]:
derived_agg = run_agg(start_date, end_date, portfolioB)

In [ ]:
portfolioB_df = get_agg_df(derived_agg.data)
portfolioB_df.head(5)

In [ ]:
port_joined_df = portfolioA_df.join(
    portfolioB_df, lsuffix=portfolioA, rsuffix=portfolioB
)
port_joined_df.rename(
    columns={
        "MarketValueUS_TECH1": "Portfolio weighted 55% to Alphabet Inc",
        "MarketValueUS_TECH2": "Portfolio weighted 55% to Netflix",
    },
    inplace=True,
)
port_joined_df.head(2)

# Step 9: Check results

Looking at the time-series of results below, we can see we were correct to allocate 55% to Google versus Netflix in first 3Q of 2019.

In [ ]:
def time_series_performance_all():
    ts_performance_all = port_joined_df.plot(
        y=[port_joined_df.columns[0], port_joined_df.columns[2]], figsize=(12, 9)
    )
    ts_performance_all.set_title(
        "NAV of portfolios from 01-01-2019 to 30-09-2019", fontsize="large"
    )
    ts_performance_all.set_ylabel("NAV in millions of $", fontsize="large")
    ts_performance_all.set_xlabel("Time: Jan to Sep 2019", fontsize="large")
    ts_performance_all.legend(prop={"size": 12})

In [ ]:
time_series_performance_all()